# ContractNLI: Qwen3-4B and Qwen3-1.7B, thinking off and on (Kaggle)

Runs the 5 scaffolds (`zeroshot`, `dspy`, `cot`, `rag`, `agentic_rag`) with thinking
**off** and **on** on the 150-instance stratified ContractNLI dev sample (seed 0), the
same instances as the Qwen3-8B n=150 results. 2 models x 2 modes x 5 scaffolds = 20 runs.

**Settings (right sidebar):** Accelerator = **GPU T4 x2**, Internet = **On**.
Add the `contractnli-kaggle-qwen-small` dataset via **+ Add Input**.

Best way to run: **Save Version -> Save & Run All (Commit)**. It runs in the background
(you can close the browser) and the results zip appears under the version's **Output**.

## 1. Install

In [ ]:
!pip install -q vllm "dspy==3.2.1" "litellm==1.73.1" sentence-transformers scikit-learn tiktoken requests
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Copy the files into the working directory

In [ ]:
import os, glob, shutil, zipfile

WORK = "/kaggle/working/contractnli"
os.makedirs(WORK, exist_ok=True)
NEEDED = ["dev.json", "run_matrix.py", "contractnli_zeroshot.py", "run_rlm_contractnli.py",
          "sample_manifest_dev_n150_seed0.csv", "run_driver.py"]

# Kaggle unpacks an uploaded zip dataset into a folder; handle a nested zip too.
for f in NEEDED:
    hits = glob.glob(f"/kaggle/input/**/{f}", recursive=True)
    if hits:
        shutil.copy(hits[0], WORK)
for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
    with zipfile.ZipFile(z) as zf:
        for n in zf.namelist():
            if os.path.basename(n) in NEEDED and not os.path.exists(f"{WORK}/{os.path.basename(n)}"):
                with open(f"{WORK}/{os.path.basename(n)}", "wb") as out:
                    out.write(zf.read(n))

missing = [f for f in NEEDED if not os.path.exists(f"{WORK}/{f}")]
assert not missing, f"missing {missing} - did you add the dataset as input?"
os.environ["CONTRACTNLI_DATA_DIR"] = WORK
print("files OK in", WORK)

## 3. Check the sample is the same 150 instances as the 8B runs

In [ ]:
import sys
sys.path.insert(0, WORK)
from run_rlm_contractnli import load_split
from run_matrix import load_manifest, sample_fingerprint

s = load_manifest(f"{WORK}/sample_manifest_dev_n150_seed0.csv", load_split("dev"))
print(len(s), sample_fingerprint(s))
assert sample_fingerprint(s) == "78850f8b821d828f", "sample differs from the 8B runs"

## 4. Serve the model(s) and run
Runs **Qwen3-1.7B only** for now: a short smoke test (thinking really off / on),
then its 10 runs. Prints progress every 2 minutes.

To run 4B afterwards, change `MODELS` to `"Qwen/Qwen3-4B"`; for both at once (one per
GPU), use `"Qwen/Qwen3-1.7B Qwen/Qwen3-4B"`. Finished instances are always skipped.

In [ ]:
MODELS = "Qwen/Qwen3-1.7B"
!cd /kaggle/working/contractnli && python -u run_driver.py --models $MODELS

## 5. Summary tables

In [ ]:
M = "Qwen/Qwen3-4B Qwen/Qwen3-1.7B"
ARGS = "--report-only --n 150 --manifest sample_manifest_dev_n150_seed0.csv --models " + M
!cd /kaggle/working/contractnli && python run_matrix.py $ARGS
!cd /kaggle/working/contractnli && python run_matrix.py $ARGS --thinking

## 6. Package the results (download from the Output tab)

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/results_qwen_small", "zip",
                    "/kaggle/working/contractnli", "results")
shutil.make_archive("/kaggle/working/logs_qwen_small", "zip",
                    "/kaggle/working/contractnli", "logs")
print("-> /kaggle/working/results_qwen_small.zip")